### Data Transformation

### Data Loading

In [0]:
dbutils.secrets.listScopes()

[SecretScope(name='kv-scope')]

In [0]:
dbutils.secrets.list('kv-scope')

[SecretMetadata(key='sp-client-id'),
 SecretMetadata(key='sp-client-secret'),
 SecretMetadata(key='sp-tenant-id')]

In [0]:
### Get the secrets

client_id = dbutils.secrets.get(scope = 'kv-scope',key = 'sp-client-id')
tenant_id = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")

In [0]:
### Set the all variables

storage_account = "bristyadlsbigdata"
source_container = "filedrop"
sink_container = "etl"


In [0]:
### Set the spark configurations

spark.conf.set(
  f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
  "OAuth"
)

spark.conf.set(
  f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
  "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
  f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
  client_id
)

spark.conf.set(
  f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
  client_secret
)

spark.conf.set(
  f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
  f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)


### Define Container Paths

In [0]:
filedrop_base_path = f"abfss://{source_container}@{storage_account}.dfs.core.windows.net/Meta/MetaInfo/"
etl_base_path = f"abfss://{sink_container}@{storage_account}.dfs.core.windows.net/Meta/MetaInfo/"

In [0]:
dbutils.fs.ls("abfss://filedrop@bristyadlsbigdata.dfs.core.windows.net/")

[FileInfo(path='abfss://filedrop@bristyadlsbigdata.dfs.core.windows.net/EPYS_Meta_RawData_Extract_20260210_20260212_from_1199.csv', name='EPYS_Meta_RawData_Extract_20260210_20260212_from_1199.csv', size=40990862, modificationTime=1772021121000),
 FileInfo(path='abfss://filedrop@bristyadlsbigdata.dfs.core.windows.net/EPYS_Meta_RawData_Extract_20260210_20260212_from_4567.csv', name='EPYS_Meta_RawData_Extract_20260210_20260212_from_4567.csv', size=40990862, modificationTime=1771933970000),
 FileInfo(path='abfss://filedrop@bristyadlsbigdata.dfs.core.windows.net/EPYS_Meta_RawData_Extract_20260210_20260212_from_4568.csv', name='EPYS_Meta_RawData_Extract_20260210_20260212_from_4568.csv', size=40990862, modificationTime=1771933960000),
 FileInfo(path='abfss://filedrop@bristyadlsbigdata.dfs.core.windows.net/Meta/', name='Meta/', size=0, modificationTime=1772368827000),
 FileInfo(path='abfss://filedrop@bristyadlsbigdata.dfs.core.windows.net/PQBV_Meta_RawData_Extract_20260210_20260212_from_4537.c

### Import Required Libraries

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime
from urllib.parse import urlparse
import os

In [0]:
date_folders = dbutils.fs.ls(filedrop_base_path)
for folder in date_folders:
    if folder.isDir():
        date_folder_name = folder.path.split("/")[-2]
        # print(f"Processing folder: {date_folder_name}")
        files = dbutils.fs.ls(folder.path)
        for file in files:
            if file.path.endswith(".csv"):
                # print(f"Processing file: {file.path}")
                # Read the CSV file
                df = spark.read.option("header", "true").option("inferSchema", "true").csv(file.path)
                # Generate current timestamp
                current_ts_string = datetime.now().strftime("%Y%m%d%H%M%S")
                # Create New etl Filename
                original_filename = os.path.basename(file.path)
                new_filename = original_filename.replace(".csv",f"_{current_ts_string}.csv")
                # Prepare Final ETL Path
                final_output_folder = etl_base_path + date_folder_name + "/"
                final_full_path = final_output_folder + new_filename
                only_folder = urlparse(etl_base_path).path.lstrip('/').rstrip('/')
                relative_path = f"{only_folder}/{date_folder_name}/{new_filename}"
                # Add Required Columns (Using lit())
                df = df.withColumn("sourcefile", lit(relative_path))
                df = df.withColumn("reportdatetime", current_timestamp())
                #  Write File (Temporary Folder)
                temp_path = final_output_folder + "temp/"
                df.coalesce(1).write.mode("overwrite").option("header", "true").csv(temp_path)
                # Rename Spark Part File
                temp_files = dbutils.fs.ls(temp_path)
                
                for f in temp_files:
                    print(f.path)
                    if f.path.endswith(".csv"):
                        dbutils.fs.mv(f.path, final_full_path)
                
                # Remove temp folder
                dbutils.fs.rm(temp_path, recurse=True)
                
                print(f"Moved to ETL: {final_full_path}")



abfss://etl@bristyadlsbigdata.dfs.core.windows.net/Meta/MetaInfo/2026-02-24/temp/_SUCCESS
abfss://etl@bristyadlsbigdata.dfs.core.windows.net/Meta/MetaInfo/2026-02-24/temp/_committed_475699121496146321
abfss://etl@bristyadlsbigdata.dfs.core.windows.net/Meta/MetaInfo/2026-02-24/temp/_started_475699121496146321
abfss://etl@bristyadlsbigdata.dfs.core.windows.net/Meta/MetaInfo/2026-02-24/temp/part-00000-tid-475699121496146321-219304d8-c51d-45ff-b8cb-79d9d473b9f8-296-1-c000.csv
Moved to ETL: abfss://etl@bristyadlsbigdata.dfs.core.windows.net/Meta/MetaInfo/2026-02-24/EPYS_Meta_RawData_Extract_20260210_20260212_from_4567_20260301124048_20260302092037.csv
abfss://etl@bristyadlsbigdata.dfs.core.windows.net/Meta/MetaInfo/2026-02-24/temp/_SUCCESS
abfss://etl@bristyadlsbigdata.dfs.core.windows.net/Meta/MetaInfo/2026-02-24/temp/_committed_6908079045673928577
abfss://etl@bristyadlsbigdata.dfs.core.windows.net/Meta/MetaInfo/2026-02-24/temp/_started_6908079045673928577
abfss://etl@bristyadlsbigdata.dfs